In [1]:
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F

import satlaspretrain_models

## Model

### Encoder

#### Foundation model

In [2]:
class FoundationModel(nn.Module):
    """
    Encoder module using a pre-trained Swin Transformer backbone with CVPT.
    Outputs multi-scale feature maps and an enhanced embedding.
    """
    def __init__(self, weights_manager, model_identifier, n_channels=3):
        super().__init__()
        # Load pre-trained Swin Transformer backbone
        self.backbone = weights_manager.get_pretrained_model(model_identifier)

        # Modify the first layer to accept n channels
        self.backbone.backbone.backbone.features[0][0] = nn.Conv2d(n_channels, 128, kernel_size=(4, 4), stride=(4, 4))

    def forward(self, x):
        # Get multi-scale feature maps from the backbone
        feature_maps = self.backbone(x)  # List: [B, 128, 128, 128], [B, 256, 64, 64], [B, 512, 32, 32], [B, 1024, 16, 16]
        embedding = feature_maps  # Last feature map as embedding
        # Enhance embedding with CVPT
        return embedding

#### CVPT

In [3]:
class CVPT(nn.Module):
    """
    Cross-Visual Prompt Tuning module to enhance the encoder's adaptability.
    Adds learnable prompts tuned with cross-attention to the feature map.
    """
    def __init__(self, num_prompts, embed_dim, num_heads, H, W):
        super().__init__()
        # Learnable prompts initialized randomly
        self.prompts = nn.Parameter(torch.randn(num_prompts, embed_dim))
        self.attention = nn.MultiheadAttention(embed_dim, num_heads)
        # Projection layer to match the feature map spatial dimensions
        self.projection = nn.Linear(num_prompts * embed_dim, embed_dim * H * W)

    def forward(self, feature_map):
        batch_size, C, H, W = feature_map.shape  # Expects [batch_size, 1024, 16, 16]
        # Flatten feature map to tokens for attention
        feature_tokens = feature_map.view(batch_size, C, H*W).permute(2, 0, 1)  # [H*W, batch_size, C]
        # Expand prompts to batch size
        prompts = self.prompts.unsqueeze(1).repeat(1, batch_size, 1)  # [num_prompts, batch_size, embed_dim]
        # Cross-attention: prompts attend to feature tokens
        attended_prompts, _ = self.attention(prompts, feature_tokens, feature_tokens)  # [num_prompts, batch_size, embed_dim]
        attended_prompts = attended_prompts.permute(1, 0, 2).contiguous().view(batch_size, -1)  # [batch_size, num_prompts * embed_dim]
        # Project back to feature map shape and add to original
        adjustment = self.projection(attended_prompts).view(batch_size, C, H, W)
        return feature_map + adjustment


#### Full Encoder

In [4]:
class Encoder(nn.Module):
    """
    Encoder module using a pre-trained Swin Transformer backbone with CVPT.
    Outputs multi-scale feature maps and an enhanced embedding.
    """
    def __init__(self, backbone_model, num_prompts, num_heads, embed_dim, H=16, W=16):
        super().__init__()
        # Load pre-trained Swin Transformer backbone
        self.backbone = backbone_model
        # CVPT module to enhance the last feature map
        self.cvpt = CVPT(num_prompts, embed_dim=embed_dim, num_heads=num_heads, H=H, W=W)

    def forward(self, x):
        # Get multi-scale feature maps from the backbone
        feature_maps = self.backbone(x)  # List: [B, 128, 128, 128], [B, 256, 64, 64], [B, 512, 32, 32], [B, 1024, 16, 16]
        feature_maps = [fm.unsqueeze(0) if fm.dim() == 3 else fm for fm in feature_maps]
        for idx, fm in enumerate(feature_maps):
            print(f'Feature map {idx}:',fm.shape)
        embedding = feature_maps[-1]  # Last feature map as embedding
        # Enhance embedding with CVPT
        enhanced_embedding = self.cvpt(embedding)
        return feature_maps, enhanced_embedding

### Decoder

#### Cross Attention

In [5]:
class CrossAttention(nn.Module):
    """
    Cross-attention module to produce task-specific embeddings for segmentation and super-resolution.
    Facilitates collaboration between tasks by attending to the shared embedding.
    """
    def __init__(self, embed_dim, num_queries_seg, num_queries_sr, num_heads):
        super().__init__()
        # Learnable queries for each task
        self.queries_seg = nn.Parameter(torch.randn(num_queries_seg, embed_dim))
        self.queries_sr = nn.Parameter(torch.randn(num_queries_sr, embed_dim))
        self.attention = nn.MultiheadAttention(embed_dim, num_heads)

    def forward(self, embedding):
        batch_size, C, H, W = embedding.shape
        # Flatten embedding to tokens
        embedding_tokens = embedding.view(batch_size, C, H*W).permute(2, 0, 1)  # [H*W, batch_size, C]
        # Segmentation task
        queries_seg = self.queries_seg.unsqueeze(1).repeat(1, batch_size, 1)  # [num_queries_seg, batch_size, embed_dim]
        seg_embedding, _ = self.attention(queries_seg, embedding_tokens, embedding_tokens)
        seg_embedding = seg_embedding.permute(1, 0, 2)  # [batch_size, num_queries_seg, embed_dim]
        # Super-resolution task
        queries_sr = self.queries_sr.unsqueeze(1).repeat(1, batch_size, 1)
        sr_embedding, _ = self.attention(queries_sr, embedding_tokens, embedding_tokens)
        sr_embedding = sr_embedding.permute(1, 0, 2)  # [batch_size, num_queries_sr, embed_dim]
        return seg_embedding, sr_embedding

#### Common Utils

In [6]:
class Down(nn.Module):
    """
    Downscaling with maxpool then double conv
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)

# Define the DoubleConv block, which consists of two convolutional layers followed by batch normalization and ReLU activation
class DoubleConv(nn.Module):
    """
    (convolution => [BN] => ReLU) * 2
    """
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

# Define the Up block, which upscales the input and concatenates it with the skip connection from the encoder, followed by a DoubleConv block
class Up(nn.Module):
    """
    Upscaling then double conv
    """
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()

         # If bilinear interpolation is used, upscale using bilinear interpolation and reduce the number of channels
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            # Otherwise, use transposed convolution for upscaling
            self.up = nn.ConvTranspose2d(in_channels , in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        # breakpoint()
        x1 = self.up(x1)
        # Calculate the difference in size between the input and the skip connection
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]

        # Pad the input to match the size of the skip connection
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])
        # Concatenate the skip connection with the upscaled input
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

# Define the final output convolution layer
class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)

In [7]:
import math

def default_conv(in_channels, out_channels, kernel_size, bias=True, groups = 1):
    wn = lambda x:torch.nn.utils.weight_norm(x)
    return nn.Conv2d(
        in_channels, out_channels, kernel_size,
        padding=(kernel_size//2), bias=bias, groups = groups)

class BasicConv(nn.Module):
    def __init__(self, in_planes, out_planes, kernel_size, stride=1, padding=1, dilation=1, groups=1, relu=True,
                 bn=False, bias=False, up_size=0,fan=False):
        super(BasicConv, self).__init__()
        wn = lambda x:torch.nn.utils.weight_norm(x)
        self.out_channels = out_planes
        self.in_channels = in_planes
        if fan:
            self.conv = nn.ConvTranspose2d(in_planes, out_planes, kernel_size=kernel_size, stride=stride, padding=padding,
                              dilation=dilation, groups=groups, bias=bias)
        else:
            self.conv = nn.Conv2d(in_planes, out_planes, kernel_size=kernel_size, stride=stride, padding=padding,
                              dilation=dilation, groups=groups, bias=bias)
        self.bn = nn.BatchNorm2d(out_planes, eps=1e-5, momentum=0.01, affine=True) if bn else None
        self.relu = nn.ReLU(inplace=True) if relu else None
        self.up_size = up_size
        self.up_sample = nn.Upsample(size=(up_size, up_size), mode='bilinear') if up_size != 0 else None

    def forward(self, x):
        x = self.conv(x)
        if self.bn is not None:
            x = self.bn(x)
        if self.relu is not None:
            x = self.relu(x)
        if self.up_size > 0:
            x = self.up_sample(x)
        return x

class Upsampler(nn.Sequential):
    def __init__(self, scale, n_feats, bn=False, act=False, bias=True):
        conv = default_conv
        m = []
        if (scale & (scale - 1)) == 0:    # Is scale = 2^n?
            print(f'number of scale: {int(math.log(scale, 2))}')
            for _ in range(int(math.log(scale, 2))):
                m.append(conv(n_feats, 4 * n_feats, 3, bias))
                m.append(nn.PixelShuffle(2))
                if bn: m.append(nn.BatchNorm2d(n_feats))

                if act == 'relu':
                    m.append(nn.ReLU(True))
                elif act == 'prelu':
                    m.append(nn.PReLU(n_feats))

        elif scale == 3:
            m.append(conv(n_feats, 9 * n_feats, 3, bias))
            m.append(nn.PixelShuffle(3))
            if bn: m.append(nn.BatchNorm2d(n_feats))

            if act == 'relu':
                m.append(nn.ReLU(True))
            elif act == 'prelu':
                m.append(nn.PReLU(n_feats))
        else:
            raise NotImplementedError

        super(Upsampler, self).__init__(*m)

#### Segmentation Head

In [8]:
class SegmentationHead(nn.Module):
    """
    Land-use segmentation head using a U-Net-like decoder with ViT-style skip connections.
    Outputs a segmentation map with the specified number of classes.
    """
    def __init__(self, n_classes, bilinear=True):
        super().__init__()
        factor = 2 if bilinear else 1
        # U-Net decoder blocks
        self.up0 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.up1 = Up(2048, 1024 // factor, bilinear)
        self.up2 = Up(1024, 512 // factor, bilinear)
        self.up3 = Up(512, 256 // factor, bilinear)
        self.up4 = Up(256, 128, bilinear)
        self.up5 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.up6 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.outc = OutConv(32, n_classes)

    def forward(self, feature_maps):
        # Start from the smallest scale and upsample with skip connections
        x = feature_maps[3]  # [batch_size, 1024, 16, 16]
        x = self.up0(x)  # [batch_size, 1024, 32, 32]
        x = self.up1(x, feature_maps[3])  # [batch_size, 512, 32, 32]
        x = self.up2(x, feature_maps[2])  # [batch_size, 256, 64, 64]
        x = self.up3(x, feature_maps[1])  # [batch_size, 128, 128, 128]
        x = self.up4(x, feature_maps[0])                   # [batch_size, 64, 256, 256]
        x = self.up5(x)                  # [batch_size, 32, 512, 512]
        x = self.up6(x)                  # [batch_size, 16, 1024, 1024]
        x = self.outc(x)                  # [batch_size, n_classes, 256, 256]
        return x

#### Super-Resolution Head

In [9]:
class SuperResolutionHead(nn.Module):
    """
    Super-resolution head that enhances the input resolution using an upsampler.
    Incorporates task-specific embedding to condition the feature map.
    """
    def __init__(self, embed_dim, num_queries, in_channels, n_channels, upscale_factor):
        super().__init__()
        conv = default_conv
        # Project task-specific embedding to match feature map channels
        self.conv = nn.Conv2d(in_channels, 64 * (upscale_factor ** 2), 3, padding=1)
        self.projector = nn.Linear(num_queries * embed_dim, in_channels)
        self.tail = nn.Sequential(
            nn.ConvTranspose2d(1024, 512, 2, 2, 0),
            nn.ConvTranspose2d(512, 256, 2, 2, 0),
            nn.ConvTranspose2d(256, 128, 2, 2, 0),
            nn.ConvTranspose2d(128, 64, 2, 2, 0),
            nn.ConvTranspose2d(64, 32, 2, 2, 0),
            Upsampler(upscale_factor, n_feats=32),
            conv(in_channels=32, out_channels=n_channels, kernel_size=3)
        )
        self.up = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 2, 2, 0),
            nn.ConvTranspose2d(64, 32, 2, 2, 0),
            Upsampler(upscale_factor, n_feats=32),
            BasicConv(in_planes=32, out_planes=n_channels, kernel_size=3, stride=1, padding=1),
        )

    def forward(self, feature_maps, sr_embedding):
        batch_size, C, H, W = feature_maps[-1].shape
        # Flatten and project super-resolution embedding
        sr_embedding_flat = sr_embedding.reshape(batch_size, -1)
        adjustment = self.projector(sr_embedding_flat).view(batch_size, C, 1, 1)
        # Enhance feature map with task-specific information
        enhanced_feature_map = feature_maps[-1] + adjustment
        res_tail = self.tail(enhanced_feature_map)
        res_up = self.up(feature_maps[0])
        return res_tail + res_up


## Main

In [10]:
num_prompts = 10
num_heads = 8
feature_map_h = 16
feature_map_w = 16
embed_dim = 1024
num_queries_seg=10
num_queries_sr= 10
n_classes = 5
upscale_factor = 3

n_channels = 13
# Dummy input (batch_size=16, 3 channels, 512x512)
input_image = torch.randn(16, n_channels, 512, 512)

In [11]:
weights_manager = satlaspretrain_models.Weights()
fm_encoder =  FoundationModel(weights_manager, "Sentinel2_SwinB_SI_RGB", n_channels)

/home/anhtn/miniconda3/envs/hungvv/lib/python3.9/site-packages/satlaspretrain_models/model.py:48: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  weights = torch.load(weights_

In [12]:
full_encoder = Encoder(backbone_model=fm_encoder, num_prompts=num_prompts, num_heads=num_heads, embed_dim=embed_dim, H=feature_map_h, W=feature_map_w)

In [13]:
cross_attention = CrossAttention(embed_dim=1024, num_queries_seg=num_queries_seg,
                                            num_queries_sr=num_queries_sr, num_heads=num_heads)

segmentation_head = SegmentationHead(n_classes=n_classes, bilinear=True)
super_resolution_head = SuperResolutionHead(embed_dim=1024, num_queries=num_queries_sr, in_channels=1024, upscale_factor=upscale_factor, n_channels=n_channels)

In [14]:
# Encode input image to get feature maps and embedding
feature_maps, embedding = full_encoder(input_image)

Feature map 0: torch.Size([16, 128, 128, 128])
Feature map 1: torch.Size([16, 256, 64, 64])
Feature map 2: torch.Size([16, 512, 32, 32])
Feature map 3: torch.Size([16, 1024, 16, 16])


In [15]:
# Generate task outputs
# Apply cross-attention to get task-specific embeddings
seg_embedding, sr_embedding = cross_attention(embedding)
print('Segmentation embedding:', seg_embedding.shape)
print('Super-resolution embedding:', sr_embedding.shape)

Segmentation embedding: torch.Size([16, 10, 1024])
Super-resolution embedding: torch.Size([16, 10, 1024])


In [16]:
hr_output = super_resolution_head(feature_maps, sr_embedding)
print(hr_output.shape)

torch.Size([16, 13, 1536, 1536])


In [17]:
seg_output = segmentation_head(feature_maps)
print(seg_output.shape)

torch.Size([16, 5, 512, 512])
